# Project 03 (final): Income prediction on real census data

**Goal:** A complete, honest ML project on a real, "messy" data set — mixed feature
types, missing values, class imbalance — including the two topics that are often
missing in practice: **choosing a threshold by costs** and a **fairness check**.

**Data:** UCI **Adult / Census Income** (48,842 people, US census 1994), via
`sklearn.datasets.fetch_openml("adult", version=2)` — no manual download needed,
scikit-learn caches the data locally (not in the repository!). Target variable: income
`>50K` vs. `<=50K` (about 24 % positives — moderately imbalanced). It contains genuine
missing values (`workclass`, `occupation`, `native-country`) and a mix of numerical and
categorical columns including sensitive attributes (`sex`, `race`) — which makes the
data set a classic for fairness discussions in ML practice.

Prior knowledge: the whole module script, project 02 (pipelines, CV, tuning), script 2.6
(imbalanced classes) and 3.1 (interpretation).

## 1. Load and explore the data

**Task:** Load the data set and take a look at the shape, the column types, the missing
values and the target distribution.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 2. Preparation: target variable and features

**Task:**
1. Build `y` as 0/1: `1` for `">50K"`, `0` for `"<=50K"`.
2. Drop `fnlwgt` (a census weighting column, not a substantive attribute) and
   `education` (redundant with `education-num`, which already carries the same
   information ordinally as a number) from the features.
3. Define two lists: `numerical_features` and `categorical_features`
   (all remaining columns, sensibly sorted by dtype).

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 3. Train/test split

**Task:** Stratified split, `test_size=0.2`, `random_state=42`. Do not touch the test
set until step 6.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 4. Preprocessing pipeline

**Task:** Build a preprocessing pipeline with `ColumnTransformer` (script 2.4):

- numerical columns: `SimpleImputer(strategy="median")` + `StandardScaler()`
  (there are no missing numerical values here, but it is a robust habit)
- categorical columns: `SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`
  (missing values such as `workclass` are replaced by the most frequent value this way;
  `sparse_output=False`, because `HistGradientBoostingClassifier` in step 6 needs dense
  rather than sparse matrices)

Combine both branches into `preprocessing = ColumnTransformer([...])`.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 5. Baseline

**Task:** Train a `DummyClassifier(strategy="most_frequent")` as a trivial point of
comparison and print its test accuracy (it simply always predicts `<=50K`).


In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 6. Model comparison under class imbalance

With about 24 % positives, **accuracy is misleading** (script 1.5/2.6) — we compare via
**PR-AUC** (`scoring="average_precision"`), which is more informative than ROC-AUC for
imbalanced classes.

**Task:** Build three pipelines (`preprocessing` + classifier), each with
`class_weight="balanced"` (script 2.6):

- `LogisticRegression(max_iter=1000, class_weight="balanced")`
- `RandomForestClassifier(random_state=42, class_weight="balanced")`
- `HistGradientBoostingClassifier(random_state=42, class_weight="balanced")`

Compare them with `StratifiedKFold(5)` + `cross_val_score(scoring="average_precision")`
on the training data and show the results as a boxplot.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 7. Tuning the best model

**Task:** Pick the best model from step 6 and tune it with `GridSearchCV`
(`cv=cv`, `scoring="average_precision"`) on the training data. A sensible grid for
`HistGradientBoostingClassifier` would be e.g. `clf__max_iter`, `clf__learning_rate`,
`clf__max_leaf_nodes` — adapt it to the model you chose.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 8. The one-off test evaluation

**Task:** `best_model = search.best_estimator_`, predictions on `X_test` at the default
threshold of 0.5, classification report, PR curve + PR-AUC on the test set.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 9. Choosing a threshold by a cost calculation

The 0.5 threshold is a convention, not a necessity (script 2.6/3.2). Scenario: an
outreach programme for financial advice wants to address people with an income of
`>50K`. A **false alarm** (FP: contacted, but actually `<=50K`) costs **50 dollars**
(wasted effort). A **missed case** (FN: `>50K`, but not contacted) costs **200 dollars**
(lost business) — four times as expensive as a false alarm.

**Task:** For a grid of thresholds (`np.linspace(0.01, 0.99, 99)`), compute the number
of FP and FN on the test set (from `y_proba`) and from those the total cost
`50 * FP + 200 * FN`. Find the cost-minimal threshold and compare it with 0.5.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Task:** Why does the cost-minimal threshold lie below 0.5, even though the model was
already trained with `class_weight="balanced"` on top of that? (Hint: script 2.6 —
class weights and threshold choice solve different problems.)

*(Your note here ...)*

## 10. Fairness check: error rates by sex

The data set contains `sex` as a feature — the model therefore has direct or indirect
(via correlated features) access to it. **Task:** At the cost-minimal threshold from
step 9, compute recall and false positive rate on the test set separately for both
groups (`Male`, `Female`) and put them side by side.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Task:** Describe the difference between the groups in 2-3 sentences. Is it explained
by the different base rate (share of `>50K`), or does a gap remain that points to a
different treatment of errors? What would you recommend to a team that wants to put
this model into operation?

*(Your note here ...)*

## 11. Permutation importance

**Task:** Compute `permutation_importance` for `best_model` **directly on the raw test
features** `X_test` (the pipeline handles the preprocessing internally — this way you
get importances per **original column**, not per one-hot dummy). Plot all features as a
horizontal bar chart.

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## Done — what you can do now

- Carry out a complete, realistic ML project end to end: mixed types, missing values,
  class imbalance
- Use `ColumnTransformer` for heterogeneous preprocessing inside one pipeline
- Compare models fairly under imbalance (PR-AUC instead of accuracy)
- Choose a decision threshold **by costs**, not by convention
- Check a model for **fairness between subgroups** before it is deployed
- Interpret importances at the level of the original features

**Reflection questions:**

1. Why is `class_weight="balanced"` alone not enough to guarantee fair error rates
   between the sex groups?
2. What would happen if you removed `sex` as a feature entirely — would that reliably
   eliminate the gap found in step 10? (Keyword: correlated proxy features such as
   `relationship` or `occupation`.)
3. The data set is from 1994. What is the danger of putting a model trained on such data
   into operation today?

**Bonus tasks:**
1. Train `HistGradientBoostingClassifier` with `categorical_features="from_dtype"`
   *without* one-hot encoding (directly on the pandas `category` columns) — compare
   PR-AUC and training time with the one-hot variant.
2. Repeat the fairness check for `race` instead of `sex`.
3. Try `CalibratedClassifierCV` and check whether the cost-minimal threshold from step 9
   shifts for the calibrated model.